# 06 — 비교 모델: Geptop 2.0 · ProteomeLM-ess (`results/compare_baselines`)

ProteomeFit 필수 유전자 분류(서비스 모델 `results/esscls_ablation/poolesm-noseq`)와 **같은 데이터 · 같은 균주별 지표 · 같은 genus 평균**으로
공개된 서열 기반 필수 유전자 예측기 2종을 평가한다. 둘 다 **필수 여부만** 예측하며, 결손 적합도(fitness) 예측은 하지 않는다.

| 비교 모델 | 입력 | 방식 | 가중치 |
|---|---|---|---|
| **Geptop 2.0** (Wen et al., *Front. Microbiol.* 10:1236, 2019, PMID 31214154) | 프로테옴 FASTA | 학습 없음 — 기준 균주와의 상호 최고 적중(RBH) · 계통 거리 가중 합 | 없음 (코드 https://github.com/RiversDong/geptop) |
| **ProteomeLM-ess** (Malbranke et al., *PNAS* 123:e2524201123, 2026, PMID 42160340, Fig. 5B) | ProteomeLM-L `hidden_states[8]` `[n, 1152]` | 학습된 2층 분류 헤드 | [`Bitbol-Lab/ProteomeLM-ess`](https://huggingface.co/Bitbol-Lab/ProteomeLM-ess) |

> [!CAUTION]
> **ProteomeLM-ess 는 우리 시험 균주 대부분을 학습에 썼다.** 공개 헤드는 OGEE v3 의 82개 유전체로 학습했는데, 거기에
> 우리 46균주 중 상당수가 **같은 균주 · 같은 필수 라벨**로 들어 있다(6절에서 확인). 그래서 ProteomeLM-ess 쪽 수치는
> genus 를 통째로 뺀 ProteomeFit 과 **같은 조건이 아니며 과대평가**다. 7절은 전체 genus 와 함께
> **ProteomeLM-ess 학습에 없는 genus 만 추린 수치**도 같이 낸다.

**조건 맞추기**

| 항목 | ProteomeFit | Geptop 2.0 | ProteomeLM-ess |
|---|---|---|---|
| 기준(참조) 정보 | 시험 · 검증 genus 밖 균주의 라벨 | **시험 genus 밖 45개 이하 균주** (원래의 DEG 37종 대신) | 공개 헤드 그대로 (학습 자료를 바꿀 수 없다) |
| 필수 라벨 | `fit_mean = -inf` | 같음 | 같음 |
| 평가 샘플 | `fit_mean` 이 공란이 아닌 195,354개 | 같음 | 같음 |
| 판정 규칙 | 보정 `conf_ess` ≥ 0.5 | 점수 > 0.24 (기본 임계값) | **균주별 상위 N개** (N = 그 균주의 실제 필수 유전자 수, 논문 Fig. 5B 방식) |
| 지표 집계 | 균주별 → genus 평균 → genus 간 평균 ± 표준편차 | 같음 | 같음 |

- 판정 규칙은 모델마다 **그 모델이 권장하는 방식**을 쓴다. ProteomeLM-ess 의 확률은 균주 간 보정이 안 돼 있어 논문도 순위 기준(top-N)을 권한다.
  top-N 은 정답 필수 유전자 수를 미리 알려 주는 셈이라 ProteomeLM-ess 에 유리하다. 임계값에 안 걸리는 **ROC AUC · PR AUC** 가 공정한 비교 지표다.
- Geptop 의 원래 기준 집합(DEG 37종)에는 *E. coli* · *Pseudomonas* 등 우리 시험 genus 가 들어 있어 genus 제외 조건이 깨진다. 그래서 46개 학습 균주로 다시 구성했다.
- ProteomeLM-ess 는 `data/emb_proteome_328m.h5` 를 그대로 입력으로 쓴다. 02 노트북이 만든 이 임베딩은 헤드가 요구하는 것과 **같은 사양**이다
  (ProteomeLM-L `hidden_states[8]` 1152차원, ESM C 600M 토큰 평균, 앞 4,096잔기, pad 만 제외한 평균). 5절에서 확인한다.

**소요 시간 · 환경** — GPU 없이 돈다.

| 단계 | 시간 | 메모 |
|---|---|---|
| 2절 BLASTP (genus 가 다른 균주 순서쌍 1,948개) | 16코어 약 10.5시간 | 끝난 쌍은 건너뛴다. NCBI BLAST+ 필요 |
| 3절 composition vector 거리 (46개 프로테옴) | 약 10분 | `work/cv_dist.csv` 에 저장해 재사용 |
| 5절 ProteomeLM-ess | 1분 미만 | 임베딩 [217,468 × 1152] float32 약 1.0 GB 를 RAM 에 올린다 |

**산출물** (`results/compare_baselines/`) — `scores.tsv` (유전자별 두 모델 점수 · 판정), `metrics_by_org.csv`, `metrics_by_genus.csv`,
`plm_ess_overlap.csv` (ProteomeLM-ess 학습 균주와의 중복), `compare.csv` (방법별 genus 평균 ± 표준편차)


In [ ]:
# !pip install biopython scikit-learn scipy pandas h5py torch safetensors huggingface_hub   # + NCBI BLAST+
import os, shutil, subprocess, itertools, json
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, h5py
from scipy.stats import wilcoxon
from sklearn.metrics import matthews_corrcoef, roc_auc_score, average_precision_score
from proteomefit import data

BASE_DIR = '../data/basedata'                    # 균주별 base data TSV (01 노트북)
PROT_H5  = '../data/emb_proteome_328m.h5'        # ProteomeLM-L hidden_states[8] [n_all, 1152] (02 노트북)
OUT_DIR  = '../results/compare_baselines'        # 산출물
WORK_DIR = os.path.join(OUT_DIR, 'work')         # FASTA · BLAST DB · BLAST 결과 (중간 파일)
BLAST    = os.path.dirname(shutil.which('blastp') or '')   # BLAST+ 실행 파일 폴더 (PATH 에 없으면 직접 지정)
N_PROC   = max(1, os.cpu_count() - 1)            # 동시에 돌릴 단일 스레드 blastp 수
CUTOFF   = 0.24                                  # Geptop 2.0 기본 임계값 (score > CUTOFF -> 필수)
ESS_HEAD = 'Bitbol-Lab/ProteomeLM-ess'           # ProteomeLM-ess 분류 헤드 (config.json + model.safetensors)
ESS_DATA = 'Bitbol-Lab/ProteomeLM-ess-data'      # 같은 논문의 학습 자료 (학습 균주 목록 data/genomes.tsv)
COLS     = ['mcc', 'roc_auc', 'pr_auc']          # 비교 지표
PROTEOMEFIT = '../results/esscls_ablation/poolesm-noseq/metrics_total_calib.csv'   # 서비스 모델 genus 별 지표

for sub in ('faa', 'db', 'hits'):
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)
print('BLAST bin:', BLAST or 'NOT FOUND', '| blastp workers:', N_PROC)


## 1. 데이터

`data.load_samples` 로 03 노트북과 같은 라벨 · 샘플을 읽는다. FASTA 서열 이름은 **전체 유전자 번호**(base data 행 번호, 0 – 217,467)라서
BLAST 결과를 라벨 배열에 바로 색인할 수 있고, 임베딩 h5 의 행 번호와도 같다. 라벨이 미상인 유전자도 프로테옴의 일부이므로 FASTA 와 점수 계산에는 넣는다.

In [ ]:
# 라벨 · 메타 [n_all = 217,468] / 평가 샘플 [n = 195,354]
D       = data.load_samples(BASE_DIR, 'fit_mean', finite_only=False)
y       = data.ess_labels(D)                                       # 평가 샘플 필수 라벨 [n] (1 = 필수)
ess_all = np.isneginf(D.yb)                                        # 전 유전자 필수 여부 [n_all] (기준 균주의 필수 목록)
org_all = D.base.org_id.to_numpy()                                 # 전 유전자 균주 [n_all]
org_ids = pd.Categorical(D.base.org_id).categories.to_numpy()      # 균주 목록 [46]
genus_of = dict(zip(D.base.org_id, D.base.org_name.str.split().str[0]))   # 균주 -> genus


def write_fasta(seqs, idx, path):
    # 단백질 서열 [m] · 전체 유전자 번호 [m] -> FASTA (">12345", 종결 기호 * 제거)
    with open(path, 'w') as f:
        for i, s in zip(idx, seqs):
            f.write(f'>{i}\n{str(s).rstrip("*")}\n')


for name in org_ids:
    idx = np.flatnonzero(org_all == name)                          # 균주의 전체 유전자 번호
    write_fasta(D.base.seq_aa.to_numpy()[idx], idx, os.path.join(WORK_DIR, 'faa', f'{name}.faa'))
print(f'genes {len(D.base):,} | eval samples {len(y):,} (essential {int(y.sum()):,}) | '
      f'strains {len(org_ids)} | genera {len(set(genus_of.values()))}')


## 2. Geptop 2.0 — BLASTP (서로 다른 genus 의 균주 순서쌍)

Geptop 은 대상 → 기준, 기준 → 대상 두 방향의 BLASTP 최고 적중이 서로를 가리키면 직교상동체(상호 최고 적중, reciprocal best hit)로 본다.
시험 genus 의 균주는 기준에서 빠지므로 같은 genus 안의 쌍은 필요 없다 (46 × 45 = 2,070 중 1,948쌍). 질의별 상위 5개 적중을 받아 첫 줄(E-value 최소)을 최고 적중으로 쓴다.
단일 스레드 blastp 여러 개를 동시에 돌리는 편이 다중 스레드 하나보다 약 3배 빠르다.

In [ ]:
def blast_exe(blast_dir, name):
    # BLAST+ 실행 파일 경로 ('blastp' -> '.../blastp' 또는 '.../blastp.exe')
    return shutil.which(name, path=blast_dir) or os.path.join(blast_dir, name)


def make_db(blast_dir, faa, db):
    # 균주 FASTA -> 단백질 BLAST DB (이미 있으면 건너뜀)
    if not os.path.exists(db + '.pin'):
        subprocess.run([blast_exe(blast_dir, 'makeblastdb'), '-in', faa, '-dbtype', 'prot', '-out', db],
                       check=True, stdout=subprocess.DEVNULL)


def run_pair(blast_dir, work_dir, q, s):
    # 질의 균주 q 전 단백질 -> 균주 s DB, 상위 5개 적중 표 (끝난 쌍은 건너뜀, .part -> 완료 시 이름 변경)
    out = os.path.join(work_dir, 'hits', f'{q}__{s}.tsv')
    if not os.path.exists(out):
        subprocess.run([blast_exe(blast_dir, 'blastp'), '-query', os.path.join(work_dir, 'faa', f'{q}.faa'),
                        '-db', os.path.join(work_dir, 'db', s), '-outfmt', '6 qseqid sseqid evalue bitscore',
                        '-max_target_seqs', '5', '-evalue', '10', '-num_threads', '1', '-out', out + '.part'], check=True)
        os.replace(out + '.part', out)
    return out


for name in org_ids:
    make_db(BLAST, os.path.join(WORK_DIR, 'faa', f'{name}.faa'), os.path.join(WORK_DIR, 'db', name))

PAIRS = [(q, s) for q, s in itertools.permutations(org_ids, 2) if genus_of[q] != genus_of[s]]   # 1,948쌍
with ThreadPoolExecutor(N_PROC) as ex:                             # subprocess 대기는 GIL 을 놓으므로 스레드로 충분
    for k, _ in enumerate(ex.map(lambda p: run_pair(BLAST, WORK_DIR, *p), PAIRS), 1):
        if k % 100 == 0 or k == len(PAIRS):
            print(f'BLAST pairs {k}/{len(PAIRS)}')


## 3. Geptop 2.0 — 계통 거리 (composition vector, $k = 6$)

균주 프로테옴의 6-mer 빈도에서 5-mer · 4-mer 로 만든 마르코프 기대 빈도 $p_0$ 를 빼 정규화한 벡터
$a(\alpha) = (f(\alpha) - p_0(\alpha)) / p_0(\alpha)$ 를 만들고, 두 벡터의 코사인으로 거리 $D = (1 - \cos) / 2$ 를 구한다.
- 예: 6-mer `MKVLAA` 가 120번 나오고 기대 빈도 $p_0$ 가 80이면 $a = (120 - 80) / 80 = 0.5$ (기대보다 50% 많다)
- 배포 코드 `geptop2.py` 의 `Distance()` 는 분모의 두 번째 제곱합에 공통 k-mer 곱을 더하는데, 원본과 같은 점수를 내기 위해 그대로 따른다
  (원본 함수와 값이 같음을 확인했다 — Kangiella–Synechococcus 거리 0.4961469597605, 벡터 원소 최대 차이 0)

In [ ]:
# Geptop 아미노산 부호 (B -> D, U -> C, X -> G, Z -> E, J -> I), 그 외 문자가 들어간 k-mer 는 버린다
AA  = {c: i for i, c in enumerate('ACDEFGHIKLMNPQRSTVWY')} | {'B': 2, 'U': 1, 'X': 5, 'Z': 3, 'J': 7}
LUT = np.full(256, -1, np.int64)
for c, i in AA.items():
    LUT[ord(c)] = i


def kmer_counts(seqs, k, lut):
    # 단백질 서열 목록 -> (k-mer 부호 정렬 [u], 개수 [u]); 부호 = 20진수 (예: 'AC' -> 0*20 + 1 = 1)
    codes = []
    for s in seqs:
        x = lut[np.frombuffer(str(s).encode(), np.uint8)]          # 잔기 부호 [L] (-1 = 무효)
        if len(x) >= k:
            win = np.lib.stride_tricks.sliding_window_view(x, k)   # [L-k+1, k]
            codes.append(win[(win >= 0).all(1)] @ (20 ** np.arange(k - 1, -1, -1)))
    return np.unique(np.concatenate(codes), return_counts=True)


def lookup(keys, vals, q):
    # 희소 사전 조회: 정렬된 keys [u] · vals [u] · 질의 q [m] -> (있음 [m], 값 [m], 없으면 0)
    pos = np.clip(np.searchsorted(keys, q), 0, len(keys) - 1)
    hit = keys[pos] == q
    return hit, np.where(hit, vals[pos], 0)


def composition_vector(seqs, lut):
    # 프로테옴 -> composition vector (6-mer 부호 [u], 값 [u]); geptop2.py CompositionVector 와 같은 값
    (k0, c0), (k1, c1), (k2, c2) = (kmer_counts(seqs, k, lut) for k in (6, 5, 4))
    s0, s1, s2 = c0.sum(), c1.sum(), c2.sum()
    n1, a = np.repeat(k1, 20), np.tile(np.arange(20), len(k1))     # 5-mer x 다음 아미노산 [20u]
    n0, n2, n3 = n1 * 20 + a, n1 % 20 ** 4 * 20 + a, n1 % 20 ** 4  # 6-mer · 뒤쪽 5-mer · 가운데 4-mer
    ok, c_n2 = lookup(k1, c1, n2)                                  # 뒤쪽 5-mer 가 있는 경우만 쓴다
    _, c_n3 = lookup(k2, c2, n3[ok])
    n0 = n0[ok]
    p0 = 1.0 * np.repeat(c1, 20)[ok] * c_n2[ok] * s0 * s2 / c_n3 / s1 / s1   # 마르코프 기대 빈도
    has0, c_n0 = lookup(k0, c0, n0)
    v = np.where(has0, (c_n0 - p0) / p0, -1.0)                     # 관측되지 않은 6-mer = -1
    order = np.argsort(n0)
    return n0[order], v[order]


def cv_distance(cv1, cv2):
    # composition vector 2개 -> 거리 (0 = 같음); geptop2.py Distance 와 같은 식
    (a, va), (b, vb) = cv1, cv2
    _, ia, ib = np.intersect1d(a, b, assume_unique=True, return_indices=True)
    prod = (va[ia] * vb[ib]).sum()                                 # 공통 k-mer 곱의 합
    only_b = np.ones(len(b), bool); only_b[ib] = False
    return (1 - prod / np.sqrt((va ** 2).sum() * (prod + (vb[only_b] ** 2).sum()))) / 2


dist_path = os.path.join(WORK_DIR, 'cv_dist.csv')
if os.path.exists(dist_path):
    DIST = pd.read_csv(dist_path, index_col=0)
else:
    CVS  = {n: composition_vector(D.base.seq_aa.to_numpy()[org_all == n], LUT) for n in org_ids}
    DIST = pd.DataFrame({b: {a: 0.0 if a == b else cv_distance(CVS[a], CVS[b]) for a in org_ids} for b in org_ids})
    DIST.to_csv(dist_path)
    del CVS
print('CV distance', DIST.shape, f'off-diagonal {DIST.values[DIST.values > 0].min():.3f} - {DIST.values.max():.3f}')


## 4. Geptop 2.0 — 점수

대상 균주마다 **다른 genus 의** 기준 균주 각각에서 상호 최고 적중을 찾고, 그 짝이 기준 균주에서 필수이면 $1/D$ 를 더한다.
대상 균주 전 유전자에서 min-max 로 0–1 점수를 만들고, 0.24 보다 크면 필수로 판정한다.

$$S_i = \mathrm{minmax}_i\Big(\sum_{j \in \mathrm{ref}} \mathbb{1}\big[\mathrm{RBH}_j(i)\ \text{is essential in}\ j\big] \cdot w_j\Big), \qquad w_j = 1 / D_j$$

- 예: 짝이 필수인 기준 균주가 거리 0.40 · 0.50 인 두 균주이면 합 = 2.5 + 2.0 = 4.5, 균주 내 최댓값이 30 이면 점수 = 0.15 → 비필수

In [ ]:
def best_hits(path):
    # BLAST 표 -> 질의 유전자 번호 -> 최고 적중 유전자 번호 Series (질의별 첫 줄 = E-value 최소)
    t = pd.read_csv(path, sep='\t', header=None, usecols=[0, 1], names=['q', 's']).drop_duplicates('q')
    return pd.Series(t.s.to_numpy(), index=t.q.to_numpy())


def geptop_scores(org_ids, genus_of, dist, ess_all, gene_org, hits_dir):
    # 전 유전자 Geptop 점수 [n_all] (균주별 min-max 0-1). gene_org = 유전자별 균주 [n_all]
    S = np.zeros(len(ess_all))
    for a in org_ids:
        for b in org_ids:
            if genus_of[a] == genus_of[b]:
                continue
            fwd = best_hits(os.path.join(hits_dir, f'{a}__{b}.tsv'))      # a 유전자 -> b 최고 적중
            rev = best_hits(os.path.join(hits_dir, f'{b}__{a}.tsv'))      # b 유전자 -> a 최고 적중
            q, s = fwd.index.to_numpy(), fwd.to_numpy()
            rbh = rev.reindex(s).to_numpy() == q                          # 상호 최고 적중 [m]
            d = dist.loc[a, b]
            S[q[rbh & ess_all[s]]] += 100.0 if d == 0 else 1.0 / d        # 짝이 필수이면 1/D 가산
        m = gene_org == a
        S[m] = (S[m] - S[m].min()) / (S[m].max() - S[m].min())            # 균주 내 min-max
    return S


GEPTOP = geptop_scores(org_ids, genus_of, DIST, ess_all, org_all, os.path.join(WORK_DIR, 'hits'))   # [n_all]
print(f'Geptop: predicted essential {int((GEPTOP > CUTOFF).sum()):,} / {len(GEPTOP):,} genes (cutoff {CUTOFF})')


## 5. ProteomeLM-ess — 공개 헤드를 우리 임베딩에 적용

공개 헤드는 `config.json` + `model.safetensors` 두 파일이고, **ProteomeLM-L `hidden_states[8]` 1152차원**을 입력으로 받는
2층 분류기다 (linear 1152 → 2048 → ReLU → dropout → linear 2048 → 2, 클래스 0 = 필수). 추론 때 dropout 은 항등이다.

$$p_i(\text{essential}) = \mathrm{softmax}\big(W_2\,\mathrm{ReLU}(W_1 h_i + b_1) + b_2\big)_0$$

- $h_i$ — 유전자 $i$ 의 ProteomeLM-L `hidden_states[8]` 벡터 (우리 `data/emb_proteome_328m.h5` 의 $i$ 번 행)
- 우리 02 노트북의 임베딩 생성 조건이 헤드의 요구 사양과 같은지는 아래에서 `assert` 로 확인한다
  (같은 체크포인트 · 같은 층 · 같은 차원). ESM C 600M 토큰 평균을 pad 만 빼고 내는 것과 앞 4,096잔기 절단도 양쪽이 같다
- 균주 하나를 통째로 넣어야 한다는 조건은 02 노트북이 프로테옴 단위로 ProteomeLM 을 돌렸으므로 이미 충족된다
- 판정은 **균주별 상위 N개**(N = 그 균주의 실제 필수 유전자 수)로, 논문 Fig. 5B 가 쓴 규칙이다

In [ ]:
from huggingface_hub import snapshot_download
from safetensors.torch import load_file


def load_ess_head(repo):
    # ProteomeLM-ess 헤드 -> (설정 dict, 파라미터 dict float32)
    folder = snapshot_download(repo, allow_patterns=['config.json', 'model.safetensors'])
    with open(os.path.join(folder, 'config.json')) as f:
        cfg = json.load(f)
    sd = {k: v.to(torch.float32) for k, v in load_file(os.path.join(folder, 'model.safetensors')).items()}
    return cfg, sd


def ess_probabilities(emb_ds, sd, chunk=20000):
    # ProteomeLM hidden_states[8] h5 데이터셋 [n_all, 1152] -> p(essential) [n_all] (softmax 클래스 0)
    out = []
    for s in range(0, emb_ds.shape[0], chunk):
        x = torch.from_numpy(emb_ds[s:s + chunk].astype(np.float32))
        h = torch.relu(x @ sd['linear_in.weight'].T + sd['linear_in.bias'])      # [chunk, 2048]
        logits = h @ sd['linear_out.weight'].T + sd['linear_out.bias']           # [chunk, 2]
        out.append(torch.softmax(logits, -1)[:, 0].numpy())
    return np.concatenate(out)


ESS_CFG, ESS_SD = load_ess_head(ESS_HEAD)
with h5py.File(PROT_H5, 'r') as f:
    # 헤드의 입력 사양과 우리 임베딩이 같은지 확인 (체크포인트 · 층 · 차원)
    assert f.attrs['proteomelm_model'] == ESS_CFG['backbone'], (f.attrs['proteomelm_model'], ESS_CFG['backbone'])
    assert int(f.attrs['layer']) == ESS_CFG['layer'] and f['emb'].shape[1] == ESS_CFG['input_dim']
    assert int(f.attrs['n_proteins']) == len(D.base)
    PLM_ESS = ess_probabilities(f['emb'], ESS_SD)                                # [n_all]

# 균주별 상위 N개 판정 (N = 그 균주의 실제 필수 유전자 수)
plm_call = np.zeros(len(PLM_ESS), int)
for name in org_ids:
    m = np.flatnonzero(org_all == name)
    n_top = int(ess_all[m].sum())
    plm_call[m[np.argsort(-PLM_ESS[m])[:n_top]]] = 1
print(f'ProteomeLM-ess: backbone {ESS_CFG["backbone"]} layer {ESS_CFG["layer"]} | '
      f'p(essential) {PLM_ESS.min():.3f} - {PLM_ESS.max():.3f} | called essential {plm_call.sum():,}')

pd.DataFrame({'org_id': org_all, 'gene_id': D.base.gene_id, 'essential': ess_all.astype(int),
              'geptop_score': GEPTOP.round(4), 'geptop_call': (GEPTOP > CUTOFF).astype(int),
              'plm_ess_p': PLM_ESS.round(6), 'plm_ess_call': plm_call}
             ).to_csv(os.path.join(OUT_DIR, 'scores.tsv'), sep='\t', index=False)


## 6. ProteomeLM-ess 학습 균주와의 중복 확인

공개 헤드의 학습 자료 목록(`Bitbol-Lab/ProteomeLM-ess-data` 의 `data/genomes.tsv`, OGEE v3 기준 89개 유전체)을 받아
우리 46균주와 겹치는 범위를 센다. NCBI 분류 개정으로 genus 이름이 다른 균주가 있어 동의어를 함께 본다.

> 겹치는 균주는 ProteomeLM-ess 가 **같은 필수 라벨을 학습에서 이미 봤다**는 뜻이다. 7절에서 전체 genus 수치와
> 겹치지 않는 genus 만의 수치를 나눠 본다.

In [ ]:
from huggingface_hub import hf_hub_download

# ProteomeLM-ess 학습 균주 중 우리 genus 와 이름이 다른 것 (NCBI 분류 개정, 균주 기호가 같아 수작업 확인)
GENUS_SYNONYM = {'Nitratidesulfovibrio': 'Desulfovibrio',    # D. vulgaris Miyazaki F
                 'Azospira': 'Dechlorosoma',                 # A. oryzae PS = D. suillum PS
                 'Stutzerimonas': 'Pseudomonas',             # S. stutzeri RCH2
                 'Sphingobacteriaceae': 'Pedobacter'}        # 'Sphingobacteriaceae bacterium GW460-11-11-14-LB5'

train = pd.read_csv(hf_hub_download(ESS_DATA, 'data/genomes.tsv', repo_type='dataset'), sep='\t')
train['genus'] = train.organism.str.split().str[0].replace(GENUS_SYNONYM)
SEEN = set(train.loc[train.role == 'cross-validation', 'genus'])        # 학습 폴드에 쓰인 genus
train[train.genus.isin(set(genus_of.values()))].to_csv(os.path.join(OUT_DIR, 'plm_ess_overlap.csv'), index=False)

ours = sorted(set(genus_of.values()))
print(f'ProteomeLM-ess 학습 유전체 {len(train)}개 (교차검증 {int((train.role == "cross-validation").sum())}개)')
print(f'우리 genus {len(ours)}개 중 학습에 포함 {len([g for g in ours if g in SEEN])}개')
print('unseen genera:', [g for g in ours if g not in SEEN])


## 7. 비교

균주별 지표 → genus 평균 → genus 간 평균 ± 표준편차. 필수 · 비필수가 모두 있는 균주만 쓴다 (ProteomeFit 과 같은 규약).
ProteomeFit 수치는 03 · 04 노트북이 저장한 genus 별 지표를 그대로 읽는다.

**주 결과는 `[main]` 표** — ProteomeLM-ess 학습에 쓰이지 않은 genus 만 평균한 값이다. 세 모델 모두 그 genus 를 학습에서 보지 못했으므로 조건이 같다.
`[reference]` 표는 전체 genus 평균이며, ProteomeLM-ess 가 나머지 genus 의 라벨을 학습에서 봤으므로 그쪽에 유리하다.

- `mcc` 는 모델마다 **자기 판정 규칙**에서 구한 값이라 규칙이 다르면 직접 비교가 어렵다
- `roc_auc` · `pr_auc` 는 임계값과 무관하므로 세 모델을 같은 잣대로 비교할 수 있다
- 필수 유전자가 없는 genus (`Magnetospirillum` · `Paraburkholderia`) 는 분류 지표가 정의되지 않아 어느 표에도 들어가지 않는다
- 참고 — ProteomeFit 의 `[main]` genus 성능은 전체 평균보다 조금 높다 (MCC 0.771 ± 0.022 대 0.725 ± 0.088). 즉 이 부분집합이 ProteomeFit 에 불리한 쪽은 아니다

In [ ]:
def genus_metrics(org, genus_of, y, score, call, cols=COLS):
    # 균주 [n] · 라벨 [n] · 점수 [n] · 판정 [n] -> (균주별 지표 [n_org, *], genus 평균 [n_genus, 3])
    rows = []
    for o in pd.unique(org):
        m = org == o
        if not 0 < y[m].sum() < m.sum():                               # 필수 · 비필수가 모두 있는 균주만
            continue
        rows.append(dict(org_id=o, genus=genus_of[o], n=int(m.sum()), n_pos=int(y[m].sum()),
                         mcc=matthews_corrcoef(y[m], call[m]), roc_auc=roc_auc_score(y[m], score[m]),
                         pr_auc=average_precision_score(y[m], score[m])))
    om = pd.DataFrame(rows)
    return om, om.groupby('genus')[cols].mean()


def load_genus_metrics(path, cols=COLS):
    # ProteomeFit metrics_total*.csv -> genus 별 지표 [n_genus, 3] (mean · median · std 요약 줄 제외)
    t = pd.read_csv(path)
    return t[pd.to_numeric(t.fold, errors='coerce').notna()].dropna(subset=cols).set_index('genus')[cols]


def summary(gm_by_method, subset=None):
    # {방법: genus 별 지표} -> 평균 ± 표준편차 표. subset 을 주면 그 genus 만 쓴다
    out = {}
    for name, gm in gm_by_method.items():
        g = gm.loc[gm.index.intersection(subset)] if subset is not None else gm
        out[name] = {c: f'{g[c].mean():.3f} ± {g[c].std():.3f}' for c in COLS} | {'n_genus': len(g)}
    return pd.DataFrame(out).T


# 균주별 · genus 별 지표 (평가 샘플 = fit_mean 이 공란이 아닌 유전자)
keep_org = D.org                                                       # 평가 샘플의 균주 [n]
om_gep, gm_gep = genus_metrics(keep_org, genus_of, y, GEPTOP[D.keep], (GEPTOP[D.keep] > CUTOFF).astype(int))
om_plm, gm_plm = genus_metrics(keep_org, genus_of, y, PLM_ESS[D.keep], plm_call[D.keep])
pd.concat([om_gep.assign(method='Geptop 2.0'), om_plm.assign(method='ProteomeLM-ess')]
          ).to_csv(os.path.join(OUT_DIR, 'metrics_by_org.csv'), index=False)

GM = {'ProteomeFit': load_genus_metrics(PROTEOMEFIT), 'Geptop 2.0': gm_gep, 'ProteomeLM-ess': gm_plm}
pd.concat({k: v for k, v in GM.items()}, names=['method']).to_csv(os.path.join(OUT_DIR, 'metrics_by_genus.csv'))

unseen   = sorted(set(genus_of.values()) - SEEN)                        # ProteomeLM-ess 학습에 없는 genus
fair_tab = summary(GM, subset=unseen)                                  # 주 결과 - 세 모델 모두에 공정
all_tab  = summary(GM)                                                 # 참고 - ProteomeLM-ess 에 유리
for name in ('Geptop 2.0', 'ProteomeLM-ess'):                          # 전체 genus 에서 ProteomeFit 과 짝 Wilcoxon 검정
    for c in COLS:
        a, b = GM['ProteomeFit'][c].align(GM[name][c], join='inner')
        all_tab.loc[name, f'p_{c}'] = f'{wilcoxon(a, b).pvalue:.1e}'

pd.concat({'unseen genera': fair_tab, 'all genera': all_tab}, names=['subset']).to_csv(os.path.join(OUT_DIR, 'compare.csv'))
seen_genus = {g: int(n) for g, n in om_plm[om_plm.genus.isin(unseen)].groupby('genus').size().items()}
print(f'[main] genera unseen by ProteomeLM-ess, strains per genus: {seen_genus}')
print(fair_tab.to_string())
print('\n[reference] all genera - ProteomeLM-ess was trained on most of them, so its numbers are optimistic')
print(all_tab.to_string())
